In [1]:
import os
import json
import pandas as pd
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

OCR_DIR = "./data/3_ocr_results"
SEG_DIR = "./data/5_telop_segments"
OUT_DIR = "./data/6_telop_instances"
FPS = 10


def _loads(x):
    return json.loads(x) if isinstance(x, str) else x


def assign_kdenlive_tracks(instances):
    """greedy: 시작시각 순으로 비어있는 트랙에 할당"""
    instances.sort(key=lambda x: (x["start_sec"], x["end_sec"]))
    track_end = []  # track_end[i] = i번 트랙의 마지막 end_sec
    for inst in instances:
        placed = False
        for ti, end in enumerate(track_end):
            if inst["start_sec"] >= end:
                inst["kdenlive_track"] = ti
                track_end[ti] = inst["end_sec"]
                placed = True
                break
        if not placed:
            inst["kdenlive_track"] = len(track_end)
            track_end.append(inst["end_sec"])
    return instances, len(track_end)

def process_one(ocr_pq, seg_pq, out_json, video_name):
    df_ocr = pd.read_parquet(ocr_pq).set_index("frame_num")
    df_seg = pd.read_parquet(seg_pq).set_index("frame_num")

    tracks = defaultdict(lambda: {"frames": set(), "text_scores": defaultdict(float)})

    for fnum, ocr_row in df_ocr.iterrows():
        if fnum not in df_seg.index:
            continue
        texts = _loads(ocr_row["ocr_texts"])
        scores = _loads(ocr_row["ocr_scores"])
        tids = _loads(df_seg.loc[fnum, "track_id"])

        for text, score, tid in zip(texts, scores, tids):
            if tid < 0:
                continue
            tracks[tid]["frames"].add(int(fnum))
            tracks[tid]["text_scores"][text] += float(score)

    instances = []
    for tid, info in tracks.items():
        frames = sorted(info["frames"])
        best_text = max(info["text_scores"].items(), key=lambda kv: kv[1])[0]
        instances.append({
            "text": best_text,
            "start_sec": round(frames[0] / FPS, 3),
            "end_sec": round((frames[-1] + 1) / FPS, 3),
        })

    # 시작시각 순 정렬 (읽기 편하고 학습에도 일관성 있음)
    instances.sort(key=lambda x: (x["start_sec"], x["end_sec"]))

    result = {
        "video": video_name,
        "instances": instances,
    }
    os.makedirs(os.path.dirname(out_json), exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

'''
def process_one(ocr_pq, seg_pq, out_json, video_name):
    df_ocr = pd.read_parquet(ocr_pq).set_index("frame_num")
    df_seg = pd.read_parquet(seg_pq).set_index("frame_num")

    # track_id → {"frames": set, "text_scores": {text: sum_score}}
    tracks = defaultdict(lambda: {"frames": set(), "text_scores": defaultdict(float)})

    for fnum, ocr_row in df_ocr.iterrows():
        if fnum not in df_seg.index:
            continue
        texts = _loads(ocr_row["ocr_texts"])
        scores = _loads(ocr_row["ocr_scores"])
        tids = _loads(df_seg.loc[fnum, "track_id"])

        for text, score, tid in zip(texts, scores, tids):
            if tid < 0:
                continue
            tracks[tid]["frames"].add(int(fnum))
            tracks[tid]["text_scores"][text] += float(score)

    instances = []
    for tid, info in tracks.items():
        frames = sorted(info["frames"])
        # score 가중 최빈값
        best_text = max(info["text_scores"].items(), key=lambda kv: kv[1])[0]
        instances.append({
            "text": best_text,
            "start_sec": round(frames[0] / FPS, 3),
            "end_sec": round((frames[-1] + 1) / FPS, 3),
        })

    instances, num_tracks = assign_kdenlive_tracks(instances)

    result = {
        "video": video_name,
        "instances": instances,
        "num_tracks": num_tracks,
    }
    os.makedirs(os.path.dirname(out_json), exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
'''
# 배치 실행
tasks = []
for channel in sorted(os.listdir(SEG_DIR)):
    seg_ch = os.path.join(SEG_DIR, channel)
    if not os.path.isdir(seg_ch):
        continue
    for fname in sorted(os.listdir(seg_ch)):
        if not fname.endswith(".parquet"):
            continue
        vn = fname[:-8]
        vn_clean = vn.rsplit("__", 1)[0]   # ← 추가: video 필드용
        seg_pq = os.path.join(seg_ch, fname)
        ocr_pq = os.path.join(OCR_DIR, channel, fname)
        out_json = os.path.join(OUT_DIR, channel, vn + ".json")   # 파일명은 원본 vn
        if not os.path.exists(ocr_pq) or os.path.exists(out_json):
            continue
        tasks.append((ocr_pq, seg_pq, out_json, vn_clean))        # JSON video 필드는 clean
        
print(f"🎯 {len(tasks)}개 처리 예정")

with ProcessPoolExecutor(max_workers=32) as pool:
    futures = [pool.submit(process_one, *t) for t in tasks]
    for f in tqdm(as_completed(futures), total=len(futures)):
        f.result()

🎯 66400개 처리 예정


  0%|          | 0/66400 [00:00<?, ?it/s]

In [1]:
import os
import json
import pandas as pd
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

OCR_DIR = "./data/3_ocr_results"
SEG_DIR = "./data/5_telop_segments"
OUT_DIR = "./data/6_telop_instances"
FPS = 10
MERGE_GAP = 0.1  # 같은 텍스트 간 이 이하의 gap이면 병합 (초)


def _loads(x):
    return json.loads(x) if isinstance(x, str) else x


def merge_same_text(instances, max_gap=MERGE_GAP):
    """같은 텍스트의 인스턴스를 시간순으로 정렬 후, gap이 0~max_gap이면 병합"""
    if not instances:
        return instances

    text_groups = defaultdict(list)
    for inst in instances:
        text_groups[inst["text"]].append(inst)

    merged = []
    for text, group in text_groups.items():
        group.sort(key=lambda x: x["start_sec"])

        current = {
            "text": text,
            "start_sec": group[0]["start_sec"],
            "end_sec": group[0]["end_sec"],
        }

        for i in range(1, len(group)):
            gap = group[i]["start_sec"] - current["end_sec"]
            if 0 <= gap <= max_gap:    # ← 여기만 변경
                current["end_sec"] = max(current["end_sec"], group[i]["end_sec"])
            else:
                merged.append(current)
                current = {
                    "text": text,
                    "start_sec": group[i]["start_sec"],
                    "end_sec": group[i]["end_sec"],
                }

        merged.append(current)

    merged.sort(key=lambda x: (x["start_sec"], x["end_sec"]))
    return merged

def process_one(ocr_pq, seg_pq, out_json, video_name):
    df_ocr = pd.read_parquet(ocr_pq).set_index("frame_num")
    df_seg = pd.read_parquet(seg_pq).set_index("frame_num")

    tracks = defaultdict(lambda: {"frames": set(), "text_scores": defaultdict(float)})

    for fnum, ocr_row in df_ocr.iterrows():
        if fnum not in df_seg.index:
            continue
        texts = _loads(ocr_row["ocr_texts"])
        scores = _loads(ocr_row["ocr_scores"])
        tids = _loads(df_seg.loc[fnum, "track_id"])

        for text, score, tid in zip(texts, scores, tids):
            if tid < 0:
                continue
            tracks[tid]["frames"].add(int(fnum))
            tracks[tid]["text_scores"][text] += float(score)

    instances = []
    for tid, info in tracks.items():
        frames = sorted(info["frames"])
        best_text = max(info["text_scores"].items(), key=lambda kv: kv[1])[0]
        instances.append({
            "text": best_text,
            "start_sec": round(frames[0] / FPS, 3),
            "end_sec": round((frames[-1] + 1) / FPS, 3),
        })

    # 같은 텍스트 + gap <= 0.5초 병합
    instances = merge_same_text(instances)

    result = {
        "video": video_name,
        "instances": instances,
    }
    os.makedirs(os.path.dirname(out_json), exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)


# 배치 실행
tasks = []
for channel in sorted(os.listdir(SEG_DIR)):
    seg_ch = os.path.join(SEG_DIR, channel)
    if not os.path.isdir(seg_ch):
        continue
    for fname in sorted(os.listdir(seg_ch)):
        if not fname.endswith(".parquet"):
            continue
        vn = fname[:-8]
        vn_clean = vn.rsplit("__", 1)[0]
        seg_pq = os.path.join(seg_ch, fname)
        ocr_pq = os.path.join(OCR_DIR, channel, fname)
        out_json = os.path.join(OUT_DIR, channel, vn + ".json")
        if not os.path.exists(ocr_pq) or os.path.exists(out_json):
            continue
        tasks.append((ocr_pq, seg_pq, out_json, vn_clean))

print(f"🎯 {len(tasks)}개 처리 예정")

with ProcessPoolExecutor(max_workers=32) as pool:
    futures = [pool.submit(process_one, *t) for t in tasks]
    for f in tqdm(as_completed(futures), total=len(futures)):
        f.result()

🎯 66400개 처리 예정


  0%|          | 0/66400 [00:00<?, ?it/s]

In [2]:
import os
import json
import pandas as pd
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
from difflib import SequenceMatcher

OCR_DIR = "./data/3_ocr_results"
SEG_DIR = "./data/5_telop_segments"
OUT_DIR = "./data/6_telop_instances"
FPS = 10
MERGE_GAP = 0.1
SIM_THRESHOLD = 0.5  # 텍스트 유사도 임계값


def _loads(x):
    return json.loads(x) if isinstance(x, str) else x


def split_track(frame_texts, fps=10, sim_threshold=SIM_THRESHOLD):
    """트랙 내에서 텍스트가 바뀌는 시점을 감지해 별도 인스턴스로 분리"""
    if not frame_texts:
        return []

    segments = []
    seg_start_idx = 0
    seg_text_scores = defaultdict(float)
    seg_text_scores[frame_texts[0][1]] += frame_texts[0][2]

    def get_best_text(ts):
        return max(ts.items(), key=lambda kv: kv[1])[0]

    for i in range(1, len(frame_texts)):
        fnum, text, score = frame_texts[i]
        best = get_best_text(seg_text_scores)
        sim = SequenceMatcher(None, text, best).ratio()

        if sim >= sim_threshold:
            seg_text_scores[text] += score
        else:
            # 텍스트 변경 감지 → 이전 세그먼트 저장
            best_text = get_best_text(seg_text_scores)
            segments.append({
                "text": best_text,
                "start_sec": round(frame_texts[seg_start_idx][0] / fps, 3),
                "end_sec": round((frame_texts[i - 1][0] + 1) / fps, 3),
            })
            seg_start_idx = i
            seg_text_scores = defaultdict(float)
            seg_text_scores[text] += score

    # 마지막 세그먼트
    best_text = get_best_text(seg_text_scores)
    segments.append({
        "text": best_text,
        "start_sec": round(frame_texts[seg_start_idx][0] / fps, 3),
        "end_sec": round((frame_texts[-1][0] + 1) / fps, 3),
    })

    return segments


def merge_same_text(instances, max_gap=MERGE_GAP):
    """같은 텍스트의 인스턴스를 시간순으로 정렬 후, gap이 0~max_gap이면 병합"""
    if not instances:
        return instances

    text_groups = defaultdict(list)
    for inst in instances:
        text_groups[inst["text"]].append(inst)

    merged = []
    for text, group in text_groups.items():
        group.sort(key=lambda x: x["start_sec"])
        current = {
            "text": text,
            "start_sec": group[0]["start_sec"],
            "end_sec": group[0]["end_sec"],
        }
        for i in range(1, len(group)):
            gap = group[i]["start_sec"] - current["end_sec"]
            if 0 <= gap <= max_gap:
                current["end_sec"] = max(current["end_sec"], group[i]["end_sec"])
            else:
                merged.append(current)
                current = {
                    "text": text,
                    "start_sec": group[i]["start_sec"],
                    "end_sec": group[i]["end_sec"],
                }
        merged.append(current)

    merged.sort(key=lambda x: (x["start_sec"], x["end_sec"]))
    return merged


def process_one(ocr_pq, seg_pq, out_json, video_name):
    df_ocr = pd.read_parquet(ocr_pq).set_index("frame_num")
    df_seg = pd.read_parquet(seg_pq).set_index("frame_num")

    # 트랙별 프레임 데이터 수집
    track_frames = defaultdict(list)

    for fnum, ocr_row in df_ocr.iterrows():
        if fnum not in df_seg.index:
            continue
        texts = _loads(ocr_row["ocr_texts"])
        scores = _loads(ocr_row["ocr_scores"])
        tids = _loads(df_seg.loc[fnum, "track_id"])

        for text, score, tid in zip(texts, scores, tids):
            if tid < 0:
                continue
            track_frames[tid].append((int(fnum), text, float(score)))

    # 각 트랙을 텍스트 변화 기준으로 분리
    instances = []
    for tid, frame_data in track_frames.items():
        frame_data.sort(key=lambda x: x[0])
        sub_instances = split_track(frame_data, FPS)
        instances.extend(sub_instances)

    # 같은 텍스트 + gap 0~0.5초 병합
    instances = merge_same_text(instances)

    result = {
        "video": video_name,
        "instances": instances,
    }
    os.makedirs(os.path.dirname(out_json), exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)


# 배치 실행
tasks = []
for channel in sorted(os.listdir(SEG_DIR)):
    seg_ch = os.path.join(SEG_DIR, channel)
    if not os.path.isdir(seg_ch):
        continue
    for fname in sorted(os.listdir(seg_ch)):
        if not fname.endswith(".parquet"):
            continue
        vn = fname[:-8]
        vn_clean = vn.rsplit("__", 1)[0]
        seg_pq = os.path.join(seg_ch, fname)
        ocr_pq = os.path.join(OCR_DIR, channel, fname)
        out_json = os.path.join(OUT_DIR, channel, vn + ".json")
        if not os.path.exists(ocr_pq) or os.path.exists(out_json):
            continue
        tasks.append((ocr_pq, seg_pq, out_json, vn_clean))

print(f"🎯 {len(tasks)}개 처리 예정")

with ProcessPoolExecutor(max_workers=32) as pool:
    futures = [pool.submit(process_one, *t) for t in tasks]
    for f in tqdm(as_completed(futures), total=len(futures)):
        f.result()

🎯 66400개 처리 예정


  0%|          | 0/66400 [00:00<?, ?it/s]